# Phase 111: HOLO4K ESM2-3B: Combine Chunks + Score

Merges Phase108/109/110's 3 partial featurization pickles, loads
Phase99's trained ensemble, runs P2Rank on the same 1000-structure
sample, and scores. Final step of the chunked-featurization fix.

## Cell 0: Install dependencies (including java, needed for P2Rank)

In [ ]:
!pip install -q torch-geometric==2.6.1 gudhi==3.10.1 biopython==1.84 transformers==4.46.3 accelerate
!apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y default-jre-headless > /dev/null 2>&1

## Cell 1: Lightweight setup: Phase51's small topology blob only

In [ ]:
import csv as _csv
import gzip, json, os, pickle, shutil, subprocess, time, urllib.request, warnings
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from Bio.PDB import PDBParser
from scipy.spatial import cKDTree
from torch_geometric.loader import DataLoader
from torch_geometric.utils import softmax as pyg_softmax
from sklearn.metrics import average_precision_score, matthews_corrcoef, roc_auc_score

warnings.filterwarnings("ignore")

AA3_TO_1={"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E","GLY":"G",
          "HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P","SER":"S",
          "THR":"T","TRP":"W","TYR":"Y","VAL":"V","MSE":"M"}

def locate(f):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(f))
    if not c: raise FileNotFoundError(f)
    return max(c, key=lambda p: p.stat().st_size)

def locate_all(f):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(f))
    return c

blob6b = torch.load(locate("phase51_graphs.pt"), map_location="cpu", weights_only=False)
TOPO_NAMES = blob6b["topo_names"]
TOPO_DIM = len(TOPO_NAMES)
assert TOPO_DIM == 34, f"expected TOPO_DIM=34, got {TOPO_DIM}"
N_PRE = 2587
ESM_DIM = 2560
if "dataset" in blob6b: del blob6b["dataset"]

GPU_OK=False
if torch.cuda.is_available():
    try:
        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")
    except RuntimeError as e: print("GPU unusable:", e)
DEVICE = torch.device("cuda" if GPU_OK else "cpu")
print(f"N_PRE={N_PRE}  TOPO_DIM={TOPO_DIM}  DEVICE={DEVICE}")

WORK = Path("/kaggle/working"); (WORK/"cache").mkdir(parents=True, exist_ok=True)
RAW = Path("/tmp/pdb_raw"); RAW.mkdir(parents=True, exist_ok=True)

## Cell 2: Sample the SAME ~1,000 HOLO4K structures (seed=2024 -- reproducible, needed for holo_paths in P2Rank)

In [ ]:
def parse_ds(url):
    lines = urllib.request.urlopen(url, timeout=60).read().decode().splitlines()
    out = []
    for ln in lines:
        ln = ln.strip()
        if not ln or ln.startswith("#"): continue
        name = ln.split("/")[-1].replace(".pdb", "")
        out.append(name[:4].upper())
    return out

holo_all = parse_ds("https://raw.githubusercontent.com/rdk/p2rank-datasets/master/holo4k.ds")
print(f"HOLO4K full: {len(holo_all)} entries")

rng = np.random.RandomState(2024)
holo_ids = sorted(holo_all); rng.shuffle(holo_ids)
holo_sample = sorted(holo_ids[:1000])
print(f"sampled {len(holo_sample)} structures (seed=2024, reproducible)")

need = holo_sample
def fetch(pid):
    for ext in ("pdb","cif"):
        p = RAW / f"{pid}.{ext}.gz"
        if p.exists() and p.stat().st_size > 500: return pid, p
    for ext in ("pdb","cif"):
        try:
            with urllib.request.urlopen(f"https://files.rcsb.org/download/{pid}.{ext}.gz", timeout=60) as r:
                b = r.read()
            if len(b) > 500: p = RAW/f"{pid}.{ext}.gz"; p.write_bytes(b); return pid, p
        except Exception: continue
    return pid, None

t0=time.time()
with ThreadPoolExecutor(max_workers=24) as ex: got = list(ex.map(fetch, need))
holo_paths = {p: f for p, f in got if f}
print(f"fetched {len(holo_paths):,}/{len(need):,} in {(time.time()-t0)/60:.1f} min")

## Cell 3: Model

In [ ]:
class EGCL(nn.Module):
    def __init__(s,h,ea=2,act=nn.SiLU,uc=True):
        super().__init__(); s.uc=uc
        s.e=nn.Sequential(nn.Linear(2*h+1+ea,h),act(),nn.Linear(h,h),act())
        s.n=nn.Sequential(nn.Linear(2*h,h),act(),nn.Linear(h,h))
        s.a=nn.Sequential(nn.Linear(h,1),nn.Sigmoid())
        if uc:
            s.c=nn.Sequential(nn.Linear(h,h),act(),nn.Linear(h,1,bias=False))
            nn.init.xavier_uniform_(s.c[-1].weight,gain=0.001)
        s.norm=nn.LayerNorm(h)
    def forward(s,h,x,ei,ea):
        src,dst=ei; rel=x[src]-x[dst]; d2=(rel**2).sum(-1,keepdim=True)
        m=s.e(torch.cat([h[src],h[dst],d2,ea],-1)); m=m*s.a(m)
        agg=torch.zeros_like(h).index_add_(0,dst,m)
        deg=torch.zeros(h.size(0),1,device=h.device).index_add_(0,dst,torch.ones_like(d2)).clamp(min=1)
        h=s.norm(h+s.n(torch.cat([h,agg/deg],-1)))
        if s.uc:
            w=s.c(m)*rel/(d2.sqrt()+1.0); x=x+torch.zeros_like(x).index_add_(0,dst,w)/deg
        return h,x

class Fus(nn.Module):
    def __init__(s,h,t):
        super().__init__(); s.p=nn.Linear(t,h); s.g=nn.Sequential(nn.Linear(h+t,h),nn.Sigmoid())
    def forward(s,h,t): return h+s.g(torch.cat([h,t],-1))*s.p(t)

class LocalizingEGNN(nn.Module):
    """Same TopologicalEGNN as every prior phase, plus a second head that
    predicts one pocket-center point per chain via a batch-aware softmax over
    a dedicated per-residue localization logit -- decoupled from the
    classification probability on purpose (see markdown above)."""
    def __init__(s, n_pre, td, h=128, L=4, dr=0.1, use_loc_head=True):
        super().__init__(); s.n_pre=n_pre; s.use_loc_head=use_loc_head
        s.embed=nn.Sequential(nn.Linear(n_pre+td,h),nn.SiLU(),nn.Linear(h,h))
        s.ls=nn.ModuleList([EGCL(h,uc=(i<L-1)) for i in range(L)])
        s.fs=nn.ModuleList([Fus(h,td) for _ in range(L)])
        s.head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Dropout(dr),nn.Linear(h//2,1))
        if use_loc_head:
            s.loc_head=nn.Sequential(nn.Linear(h,h//2),nn.SiLU(),nn.Linear(h//2,1))

    def forward(s, d):
        h=s.embed(d.x); pos=d.pos; t=d.x[:,s.n_pre:]
        for l,f in zip(s.ls,s.fs): h,pos=l(h,pos,d.edge_index,d.edge_attr); h=f(h,t)
        logits = s.head(h).squeeze(-1)
        pred_center = None
        if s.use_loc_head:
            batch = d.batch if hasattr(d, "batch") and d.batch is not None \
                    else torch.zeros(h.size(0), dtype=torch.long, device=h.device)
            loc_logit = s.loc_head(h).squeeze(-1)
            attn = pyg_softmax(loc_logit, batch)                    # sums to 1 per graph
            weighted = attn.unsqueeze(-1) * pos                     # (N,3)
            n_graphs = int(batch.max().item()) + 1
            pred_center = torch.zeros(n_graphs, 3, device=h.device).index_add_(0, batch, weighted)
        return logits, pred_center

## Cell 4: Load Phase99's 3 trained checkpoints

In [ ]:
ensemble = []
for seed in (0, 1, 2):
    m = LocalizingEGNN(N_PRE, TOPO_DIM, use_loc_head=True).to(DEVICE)
    ckpt = locate(f"holo_clean_deploy_esm3b_seed{seed}.pt")
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    m.eval()
    ensemble.append(m)
    print(f"  loaded seed {seed} from {ckpt}")
print(f"ensemble of {len(ensemble)} trained models ready")

## Cell 5: Merge Phase108/109/110's 3 partial featurization pickles into `built` (numpy -> torch)

In [ ]:
TENSOR_KEYS = {"x", "pos", "edge_index", "edge_attr"}
built = {}
n_total = 0
for chunk_idx, phase_num in enumerate((108, 109, 110)):
    matches = [p for p in locate_all(f"phase{phase_num}_raw_features_chunk{chunk_idx}.pkl")]
    if not matches:
        raise FileNotFoundError(f"chunk {chunk_idx} pickle not found -- is Phase{phase_num} attached as a kernel_source?")
    p = max(matches, key=lambda x: x.stat().st_size)
    with open(p, "rb") as fh:
        chunk = pickle.load(fh)
    for k, v in chunk.items():
        built[k] = {kk: (torch.from_numpy(vv) if kk in TENSOR_KEYS else vv) for kk, vv in v.items()}
    print(f"  chunk {chunk_idx} ({p}): {len(chunk)} chains")
    n_total += len(chunk)
print(f"merged {len(built)}/{n_total} chains total")

## Cell 6: P2Rank on the same 1,000 structures (identical to Phase 24/33/39/54/94/100)

In [ ]:
t0=time.time(); PRANK=None
subprocess.run("curl -sL https://github.com/rdk/p2rank/releases/download/2.4.2/p2rank_2.4.2.tar.gz "
               "-o /tmp/p2.tar.gz && tar xzf /tmp/p2.tar.gz -C /tmp", shell=True, check=True, timeout=1200)
PRANK = str(next(Path("/tmp").glob("p2rank_*/prank"))); os.chmod(PRANK, 0o755)

pW = Path("/tmp/p2_111"); shutil.rmtree(pW, ignore_errors=True); pW.mkdir()
pOUT = Path("/tmp/p2_111_out"); shutil.rmtree(pOUT, ignore_errors=True)
for pid in sorted(holo_paths):
    plain = pW / f"{pid}.pdb"
    if not plain.exists():
        with gzip.open(holo_paths[pid],"rb") as fi, open(plain,"wb") as fo: shutil.copyfileobj(fi,fo)
(pW/"ds.ds").write_text("\n".join(str(pW/f"{p}.pdb") for p in sorted(holo_paths))+"\n")

r = subprocess.run([PRANK,"predict",str(pW/"ds.ds"),"-o",str(pOUT),
                    "-threads", str(max(1, os.cpu_count() or 2))],
                   capture_output=True, text=True, timeout=7200)
print(f"P2Rank rc={r.returncode} in {(time.time()-t0)/60:.1f} min")
pred_csvs = {p.stem.replace(".pdb_predictions",""): p for p in pOUT.rglob("*_predictions.csv")}
print(f"P2Rank outputs: {len(pred_csvs)}/{len(holo_paths)}")

def p2rank_top_center(pid):
    f = pred_csvs.get(pid)
    if f is None: return None
    with open(f) as fh:
        rows = list(_csv.DictReader(fh))
    if not rows: return None
    rows = [{k.strip(): v.strip() if isinstance(v,str) else v for k,v in row.items() if k} for row in rows]
    cx = next((k for k in rows[0] if k.lower().replace("_","")=="centerx"), None)
    cy = next((k for k in rows[0] if k.lower().replace("_","")=="centery"), None)
    cz = next((k for k in rows[0] if k.lower().replace("_","")=="centerz"), None)
    if not (cx and cy and cz): return None
    try:
        return np.array([float(rows[0][cx]), float(rows[0][cy]), float(rows[0][cz])], dtype=np.float32)
    except (ValueError, KeyError):
        return None

p2rank_centers = {pid: p2rank_top_center(pid) for pid in holo_paths}
print(f"P2Rank centers parsed: {sum(1 for v in p2rank_centers.values() if v is not None)}/{len(holo_paths)}")

## Cell 7: Score, compare to literature AND to the 35M-vs-3B comparison

In [ ]:
mu, sd = blob6b["topo_mean"], blob6b["topo_std"]

@torch.no_grad()
def predict_chain(entry):
    x = entry["x"].clone()
    x[:, N_PRE:] = (x[:, N_PRE:] - torch.from_numpy(mu)) / torch.from_numpy(sd)
    d = type("D", (), {})()
    d.x=x.to(DEVICE); d.pos=entry["pos"].to(DEVICE)
    d.edge_index=entry["edge_index"].to(DEVICE); d.edge_attr=entry["edge_attr"].to(DEVICE)
    d.batch = torch.zeros(x.size(0), dtype=torch.long, device=DEVICE)
    seed_logits, seed_centers = [], []
    for m in ensemble:
        lg, pc = m(d)
        seed_logits.append(torch.sigmoid(lg).cpu().numpy())
        seed_centers.append(pc.cpu().numpy()[0])
    offset = entry["ca"].mean(axis=0)
    return float(np.mean(seed_logits, axis=0).sum()), np.mean(seed_centers, axis=0) + offset


def dcc_dca(pred_center, ligs):
    if pred_center is None or not ligs: return None, None
    best_dcc, best_dca = np.inf, np.inf
    for lig_atoms in ligs:
        if len(lig_atoms) == 0: continue
        com = lig_atoms.mean(axis=0)
        d_ = float(np.linalg.norm(pred_center - com))
        if d_ < best_dcc:
            best_dcc = d_
            best_dca = float(np.linalg.norm(lig_atoms - pred_center, axis=1).min())
    return (best_dcc if np.isfinite(best_dcc) else None, best_dca if np.isfinite(best_dca) else None)


by_structure = defaultdict(list)
for (pid,cid), entry in built.items():
    by_structure[pid].append((cid, entry))

ours_dcc, ours_dca, p2_dcc, p2_dca = [], [], [], []
for pid, chain_list in by_structure.items():
    ligs = chain_list[0][1]["ligs"]
    best_conf, best_center = -np.inf, None
    for cid, entry in chain_list:
        conf, center = predict_chain(entry)
        if conf > best_conf: best_conf, best_center = conf, center
    d1, a1 = dcc_dca(best_center, ligs)
    if d1 is not None: ours_dcc.append(d1); ours_dca.append(a1)

    c_p2 = p2rank_centers.get(pid)
    if c_p2 is not None:
        d2, a2 = dcc_dca(c_p2, ligs)
        if d2 is not None: p2_dcc.append(d2); p2_dca.append(a2)

ours_dcc_rate = float(np.mean([1.0 if d<4.0 else 0.0 for d in ours_dcc]))
ours_dca_rate = float(np.mean([1.0 if a<4.0 else 0.0 for a in ours_dca]))
p2_dcc_rate = float(np.mean([1.0 if d<4.0 else 0.0 for d in p2_dcc])) if p2_dcc else float("nan")
p2_dca_rate = float(np.mean([1.0 if a<4.0 else 0.0 for a in p2_dca])) if p2_dca else float("nan")

LIT = {"fpocket": (0.192, 0.457), "P2Rank": (0.314, 0.621), "EquiPocket": (0.337, 0.662)}

print(f"{'method':<32}{'DCC':>8}{'DCA':>8}{'n':>6}")
print("-"*54)
for name,(d,a) in LIT.items():
    print(f"{name+' (literature, full 4001)':<32}{d:>8.3f}{a:>8.3f}{'4001':>6}")
print(f"{'P2Rank (this run, 1000-sample)':<32}{p2_dcc_rate:>8.3f}{p2_dca_rate:>8.3f}{len(p2_dcc):>6}")
print(f"{'OURS (clean train, 1000-sample)':<32}{ours_dcc_rate:>8.3f}{ours_dca_rate:>8.3f}{len(ours_dcc):>6}")

print(f"\nvs literature EquiPocket : DCC {ours_dcc_rate-0.337:+.4f}  DCA {ours_dca_rate-0.662:+.4f}")
print(f"vs literature P2Rank     : DCC {ours_dcc_rate-0.314:+.4f}  DCA {ours_dca_rate-0.621:+.4f}")
print(f"vs this-run P2Rank       : DCC {ours_dcc_rate-p2_dcc_rate:+.4f}  DCA {ours_dca_rate-p2_dca_rate:+.4f}")

verdict = ("BEATS literature EquiPocket on a large, unfiltered HOLO4K sample -- "
          "consistent with the COACH420 finding" if ours_dcc_rate > 0.337 else
          "beats P2Rank-literature but not EquiPocket" if ours_dcc_rate > 0.314 else
          "does not beat published baselines on this sample")
print(f"\nVERDICT: {verdict}")

json.dump({"ours": {"dcc": ours_dcc_rate, "dca": ours_dca_rate, "n": len(ours_dcc)},
           "p2rank_this_run": {"dcc": p2_dcc_rate, "dca": p2_dca_rate, "n": len(p2_dcc)},
           "literature": LIT, "n_train_excluded": 7513-7061, "n_sample": len(holo_sample), "verdict": verdict},  # from Phase99's own log
          open(WORK/"phase111_summary.json", "w"), indent=2)
print("\nsaved -> phase111_summary.json")

In [ ]:
print(f"\n--- HOLO4K DCC: ESM2 capacity lever at round-3 scale ---")
print(f"{'embedding':<28}{'hidden':>8}{'DCC (3-seed)':>14}{'DCC (8-seed)':>14}{'gap to P2Rank':>16}")
print("-"*82)
print(f"{'ESM2-35M (Phase54/56)':<28}{480:>8}{0.6483:>14.4f}{0.6453:>14.4f}{-0.0127:>16.4f}")
print(f"{'ESM2-3B (this notebook)':<28}{2560:>8}{ours_dcc_rate:>14.4f}{'pending':>14}"
      f"{ours_dcc_rate-p2_dcc_rate:>16.4f}")
print(f"\ntraining pool size: 7,061 clean chains (from Phase99)")
print(f"this-run P2Rank DCC: {p2_dcc_rate:.4f}")